# Jumper: GPU training and simulation in Colab

[README](https://github.com/KingKongRobotics/jumper/blob/main/notebooks/README.md) · [中文](https://github.com/KingKongRobotics/jumper/blob/main/notebooks/README.zh.md)

Select **Runtime → Change runtime type → GPU**. Your computer only needs a browser.

**First run:** settings → environment checks → optional demo/smoke → training → progress → simulation → export/backup.

**Continue:** read backup → matching environment → restore checks → more training → progress → simulation → export/backup.

Choose a mode, then run the cells in order. Demo and smoke cells can be skipped.
Drive snapshots are optional; download the backup before Colab discards `/content`.
A completed run or an ONNX export does not establish a converged gait or physical-robot behavior.


## 1. Choose the run

For a new run, choose a task and budget. For continuation, the backup supplies the task and environment count; **ITERATIONS** is the additional update budget.
Re-running this form preserves the separately stored backup/checkpoint selection.


In [ ]:
# @title Run settings
import json
import math
import re
from pathlib import Path

WORKFLOW_MODE = "New training"  # @param ["New training", "Continue training"]
TASK = "jumper.tripod"  # @param ["jumper.flat", "jumper.tripod", "jumper.tetrapod", "jumper.ripple", "jumper.posture", "jumper.five_foot", "jumper.dance", "jumper.jump", "jumper.ref_free_jump", "jumper.swing"]
MODEL = "jumper"
NUM_ENVS = 256  # @param {type:"integer"}
ITERATIONS = 500  # @param {type:"integer"}
USE_DRIVE = False  # @param {type:"boolean"}
DOWNLOAD_FILES = True  # @param {type:"boolean"}

RUN_STATE = globals().setdefault("_JUMPER_RUN_STATE", {})
if WORKFLOW_MODE not in ("New training", "Continue training"):
    raise ValueError("Choose New training or Continue training")
if any(isinstance(v, bool) or not isinstance(v, int) or v < 1 for v in (NUM_ENVS, ITERATIONS)):
    raise ValueError("NUM_ENVS and ITERATIONS must be positive integers")
if RUN_STATE.get("backup_config") and WORKFLOW_MODE == "Continue training":
    saved = RUN_STATE["backup_config"]
    TASK, MODEL, NUM_ENVS = saved["task"], saved["model"], saved["num_envs"]
print("Mode:", WORKFLOW_MODE, "task:", TASK, "environments:", NUM_ENVS,
      "additional updates:" if WORKFLOW_MODE == "Continue training" else "updates:", ITERATIONS)
if RUN_STATE.get("restored_checkpoint"):
    print("Preserved checkpoint:", RUN_STATE["restored_checkpoint"])


### Advanced source and optional checks

New backups record the repository, commit and core versions. Older backups may require the repository URL below.
Only enable **START_NEW_RUN** to deliberately discard a previous recovery selection; saved files are retained.


In [ ]:
# @title Advanced source and optional checks
REPOSITORY_URL = "https://github.com/KingKongRobotics/jumper.git"  # @param {type:"string"}
SOURCE_REVISION = "main"  # @param {type:"string"}
CHECKOUT_FOLDER = "jumper"  # @param {type:"string"}
CORE_VERSION_OVERRIDES = ""  # @param {type:"string"}
DRIVE_FOLDER = "JumperColab"  # @param {type:"string"}
RUN_DEMO = False  # @param {type:"boolean"}
RUN_SMOKE = True  # @param {type:"boolean"}
START_NEW_RUN = False  # @param {type:"boolean"}

CORE_PACKAGES = {"torch", "mujoco", "mujoco-warp", "warp-lang", "numpy", "tensordict"}
VERSION_PATTERN = r"[0-9]+(?:\.[0-9]+)+(?:[a-zA-Z]+[0-9]*)?(?:\+[A-Za-z0-9_.-]+)?"
def core_versions(value):
    if not isinstance(value, dict) or set(value) - CORE_PACKAGES:
        raise ValueError("Core versions must be a mapping of the six allowed packages")
    if any(not isinstance(v, str) or not re.fullmatch(VERSION_PATTERN, v) for v in value.values()):
        raise ValueError("Core versions must be exact versions, without URLs or pip flags")
    if "torch" in value and value["torch"].split("+")[0] != "2.9.1":
        raise ValueError("This notebook supports the validated torch 2.9.1 / torchvision 0.24.1 pair")
    return dict(value)

if START_NEW_RUN:
    if WORKFLOW_MODE != "New training":
        raise ValueError("START_NEW_RUN requires New training mode")
    for key in ("backup_config", "backup_archive", "backup_manifest", "restored_checkpoint", "previous_backup_selection"):
        RUN_STATE.pop(key, None)
    print("Recovery selection cleared; existing files remain available.")
CORE_VERSIONS = core_versions(json.loads(CORE_VERSION_OVERRIDES)) if CORE_VERSION_OVERRIDES.strip() else {}
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", CHECKOUT_FOLDER):
    raise ValueError("CHECKOUT_FOLDER must be one simple folder name under /content")
if not SOURCE_REVISION.strip() or SOURCE_REVISION.startswith("-"):
    raise ValueError("Enter a branch, tag or commit as SOURCE_REVISION")


## 2. Read a backup before installing

New runs skip this cell automatically. Continuation reads a ZIP or a published Drive checkpoint and selects its recorded commit, task, batch and core versions.
This bounded metadata read does not execute checkpoint code. Full archive/path/CRC/hash validation follows after installation.
To select another backup, enable **REPLACE_BACKUP_SELECTION**. A failed selection leaves the previous one intact.


In [ ]:
# @title Continue training: backup selection
import stat
import uuid
import zipfile
from pathlib import PurePosixPath

BACKUP_SOURCE = "Upload ZIP"  # @param ["Upload ZIP", "Drive checkpoint"]
DRIVE_CHECKPOINT = ""  # @param {type:"string"}
REPLACE_BACKUP_SELECTION = False  # @param {type:"boolean"}

def read_backup_metadata(archive):
    with zipfile.ZipFile(archive) as package:
        entries = package.infolist()
        if len(entries) > 10000 or sum(e.file_size for e in entries) > 10 * 1024**3:
            raise ValueError("Backup exceeds the member or expanded-size limit")
        seen = set()
        for entry in entries:
            name = entry.filename
            if entry.orig_filename != name:
                raise ValueError("Truncated backup member name")
            path = PurePosixPath(name)
            if (not name or "\\" in name or path.is_absolute() or ":" in name
                    or ".." in path.parts or path.as_posix() != name.rstrip("/")
                    or name.rstrip("/") in seen
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError("Unsafe or duplicate backup member: " + name)
            seen.add(name.rstrip("/"))
        info = package.getinfo("run/colab-run.json")
        if info.file_size > 1024**2 or info.flag_bits & 1:
            raise ValueError("Unsupported or oversized backup manifest")
        return json.loads(package.read(info))

def validate_backup_metadata(manifest):
    if not isinstance(manifest, dict) or type(manifest.get("schema")) is not int or manifest["schema"] != 1:
        raise ValueError("Unsupported backup manifest")
    cfg = manifest.get("config")
    if not isinstance(cfg, dict):
        raise ValueError("Backup has no recorded training configuration")  # noqa: TRY004
    for key in ("task", "model"):
        if not isinstance(cfg.get(key), str) or not re.fullmatch(r"[A-Za-z0-9_]+(?:\.[A-Za-z0-9_]+)*", cfg[key]):
            raise ValueError("Invalid backup " + key)
    if not isinstance(cfg.get("revision"), str) or not re.fullmatch(r"[a-fA-F0-9]{40}", cfg["revision"]):
        raise ValueError("Backup must record one exact source commit")
    if (isinstance(cfg.get("num_envs"), bool) or not isinstance(cfg.get("num_envs"), int)
            or not 1 <= cfg["num_envs"] <= 1000000):
        raise ValueError("Invalid recorded environment count")
    if cfg.get("backend") != "warp" or cfg.get("device") != "cuda:0":
        raise ValueError("This notebook continues its recorded Warp CUDA runs only")
    environment = cfg.get("mjrl_environment", {})
    if (not isinstance(environment, dict) or len(environment) > 100
            or any(not isinstance(k, str) or not re.fullmatch(r"MJRL_[A-Z0-9_]+", k)
                   or not isinstance(v, str) or len(v) > 4096 or "\x00" in v for k, v in environment.items())):
        raise ValueError("Invalid recorded MJRL environment settings")
    versions = core_versions(cfg.get("runtime_versions"))
    if set(versions) != CORE_PACKAGES:
        raise ValueError("Backup must record all six core package versions")
    repo = cfg.get("repository_url")
    if repo is not None and (not isinstance(repo, str) or not re.fullmatch(
            r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", repo)):
        raise ValueError("Backup repository must be an HTTPS GitHub repository URL")
    if not manifest.get("checkpoints"):
        raise ValueError("Backup has no saved checkpoints")
    return dict(cfg)

if WORKFLOW_MODE == "Continue training":
    if not RUN_STATE.get("backup_config") or REPLACE_BACKUP_SELECTION:
        candidate_archive = None
        candidate_checkpoint = None
        if BACKUP_SOURCE == "Upload ZIP":
            from google.colab import files
            uploaded = files.upload()
            if len(uploaded) != 1:
                raise ValueError("Upload exactly one notebook backup ZIP")
            _, payload = next(iter(uploaded.items()))
            inbox = Path("/content/jumper-backups")
            inbox.mkdir(parents=True, exist_ok=True)
            candidate_archive = inbox / (uuid.uuid4().hex + ".zip")
            candidate_archive.write_bytes(payload)
            candidate_manifest = read_backup_metadata(candidate_archive)
            del uploaded, payload
        elif BACKUP_SOURCE == "Drive checkpoint":
            from google.colab import drive
            drive.mount("/content/drive")
            candidate_checkpoint = Path(DRIVE_CHECKPOINT).resolve()
            if not candidate_checkpoint.is_relative_to(Path("/content/drive/MyDrive")):
                raise ValueError("Choose a published checkpoint under MyDrive")
            if any(part.startswith(".pending-") for part in candidate_checkpoint.parts):
                raise ValueError("Choose a completed Drive snapshot, not a pending upload")
            if not candidate_checkpoint.is_file():
                raise FileNotFoundError(candidate_checkpoint)
            metadata_path = candidate_checkpoint.parent / "colab-run.json"
            if metadata_path.stat().st_size > 1024**2:
                raise ValueError("Oversized Drive manifest")
            candidate_manifest = json.loads(metadata_path.read_text())
        else:
            raise ValueError("Choose Upload ZIP or Drive checkpoint")
        candidate_config = validate_backup_metadata(candidate_manifest)
        if RUN_STATE.get("backup_config") and "previous_backup_selection" not in RUN_STATE:
            RUN_STATE["previous_backup_selection"] = {key: RUN_STATE.get(key) for key in (
                "backup_config", "backup_manifest", "backup_archive", "restored_checkpoint")}
        RUN_STATE.update(backup_config=candidate_config, backup_manifest=candidate_manifest,
                         backup_archive=str(candidate_archive) if candidate_archive else None,
                         restored_checkpoint=str(candidate_checkpoint) if candidate_checkpoint else None)
    saved = RUN_STATE["backup_config"]
    TASK, MODEL, NUM_ENVS = saved["task"], saved["model"], saved["num_envs"]
    SOURCE_REVISION = saved["revision"]
    CORE_VERSIONS = core_versions(saved["runtime_versions"])
    if saved.get("repository_url"):
        REPOSITORY_URL = saved["repository_url"]
    else:
        print("Older backup: use Advanced source to select a repository containing this commit.")
    print("Continue:", TASK, "environments:", NUM_ENVS, "commit:", SOURCE_REVISION)
    print("Repository:", REPOSITORY_URL, "core versions:", CORE_VERSIONS)
else:
    print("New training: no backup upload is needed.")


## 3. Match and check the environment

The requested source resolves to one commit. Setup reuses a matching checkout and interpreter without resetting files.
It checks an attached GPU before downloading, then verifies real CUDA, Warp and NVIDIA EGL execution.


In [ ]:
if WORKFLOW_MODE == "Continue training":
    if not RUN_STATE.get("backup_config"):
        raise RuntimeError("Read a backup in section 2 before setting up continuation")
    saved = RUN_STATE["backup_config"]
    TASK, MODEL, NUM_ENVS = saved["task"], saved["model"], saved["num_envs"]
    SOURCE_REVISION = saved["revision"]
    CORE_VERSIONS = core_versions(saved["runtime_versions"])
    REPOSITORY_URL = saved.get("repository_url", REPOSITORY_URL)
    import os

    for key in tuple(os.environ):
        if key.startswith("MJRL_"):
            os.environ.pop(key)
    os.environ.update(saved.get("mjrl_environment", {}))
elif RUN_STATE.get("backup_config"):
    raise RuntimeError("Recovered backup is preserved. Choose Continue training or explicitly enable START_NEW_RUN.")
import json
import os
import signal
import subprocess
import sys
import uuid
import venv
from collections import deque
from pathlib import Path

if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Jumper supports Python 3.10–3.13; select a compatible Colab runtime")
if not Path("/content").is_dir():
    raise RuntimeError("This notebook is intended for Google Colab with /content available")

def run(command, *, cwd=None):
    print("+", " ".join(str(part) for part in command), flush=True)
    command = [str(part) for part in command]
    tail = deque(maxlen=60)
    process = subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line[-4096:])
        returncode = process.wait()
        output = "".join(tail)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command, output=output)
        return subprocess.CompletedProcess(command, returncode, stdout=output)
    except BaseException:
        if process.poll() is None:
            # Interrupt the whole subprocess group so helper cleanup can save checkpoints.
            try:
                os.killpg(process.pid, signal.SIGINT)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
        raise
    finally:
        process.stdout.close()

# Fail before downloads when no GPU was attached to this runtime.
run(["nvidia-smi", "-L"])
gpu_rows = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    text=True,
).strip().splitlines()
if not gpu_rows:
    raise RuntimeError("No GPU detected; change the Colab runtime type to GPU")
gpu_name, capability = [part.strip() for part in gpu_rows[0].rsplit(",", 1)]
capability = float(capability)
if capability < 7.5:
    raise RuntimeError(f"Unsupported GPU for this notebook's tested wheel family: {gpu_name}")
print("GPU:", gpu_name, "compute capability:", capability)

REPO = Path("/content") / CHECKOUT_FOLDER
created_checkout = not REPO.exists()
if created_checkout:
    run(["git", "clone", REPOSITORY_URL, REPO])
else:
    if not (REPO / ".git").exists():
        raise RuntimeError(f"{REPO} already exists and is not a Git checkout")
    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO, text=True,
    ).strip()
    if origin.rstrip("/").removesuffix(".git") != REPOSITORY_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Existing checkout origin differs: {origin}")

# Fetching updates references; no checkout content is overwritten.
run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO)
COMMIT = subprocess.check_output(
    ["git", "rev-parse", "FETCH_HEAD^{commit}"], cwd=REPO, text=True,
).strip()
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if head != COMMIT:
    # A freshly cloned clean checkout may be pinned. Reused checkouts are never switched.
    status = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True)
    if not created_checkout or status:
        raise RuntimeError(f"Existing checkout is {head}, requested {COMMIT}; use a new runtime")
    run(["git", "checkout", "--detach", COMMIT], cwd=REPO)
if not (REPO / "tools/colab.py").is_file():
    raise RuntimeError(
        f"Revision {COMMIT} does not contain this notebook's Colab helper. "
        "Choose the accompanying release/fork revision, then rerun in a fresh runtime."
    )
print("Pinned source commit:", COMMIT)

play_source = (REPO / "scripts/play.py").read_text()
required_source_flags = ("--checkpoint-command-ranges", "--command", "--evaluation-out", "--replay-info-out")
if any(flag not in play_source for flag in required_source_flags):
    raise RuntimeError("This recorded source uses the older workflow. Open its matching notebook; do not change the recorded commit to force continuation.")

PYTHON = REPO / ".venv/bin/python"
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(REPO / ".venv")
run([PYTHON, "-c", "import sys; print('isolated interpreter:', sys.executable, sys.prefix)"])
# Colab can omit ensurepip. Host pip manages only the explicitly selected venv.
# This also repairs a previously created interpreter that has no pip yet.
kernel_pip_version = subprocess.check_output(
    [sys.executable, "-m", "pip", "--version"], text=True,
).split()[1]
if tuple(int(part) for part in kernel_pip_version.split(".")[:2]) < (22, 3):
    raise RuntimeError("The Colab kernel needs pip 22.3+ for --python; select a newer runtime")
run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "--upgrade", "pip"])
SESSION = Path(RUN_STATE.setdefault("session", str(Path("/content/jumper-colab") / uuid.uuid4().hex)))
SESSION.mkdir(parents=True, exist_ok=True)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"


In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "ffmpeg"])
run([PYTHON, "-m", "pip", "install", "--upgrade", "pip"])
TORCH_INDEX = "cu126" if capability < 8.0 else "cu128"
requested_torch = CORE_VERSIONS.get("torch", "2.9.1")
if "+" in requested_torch and requested_torch != f"2.9.1+{TORCH_INDEX}":
    raise RuntimeError(
        f"Saved Torch wheel {requested_torch} needs a compatible GPU; this GPU selects {TORCH_INDEX}"
    )
run([
    PYTHON, "-m", "pip", "install", "torch==2.9.1", "torchvision==0.24.1",
    "--index-url", f"https://download.pytorch.org/whl/{TORCH_INDEX}",
])
run([PYTHON, "-m", "pip", "install", "-e", str(REPO)])
# Restore only explicitly allowed exact versions; no shell, URLs or pip flags are accepted.
core_requirements = [f"{name}=={version}" for name, version in sorted(CORE_VERSIONS.items())
                     if name != "torch"]
if core_requirements:
    run([PYTHON, "-m", "pip", "install", *core_requirements])
run([PYTHON, "-m", "pip", "install", "matplotlib"])
run([PYTHON, "-m", "pip", "check"])

# Colab may expose NVIDIA's driver without registering its EGL vendor with GLVND.
# Keep the override in this session; never replace a system vendor configuration.
def configure_nvidia_egl(session, library_candidates):
    library = next((
        Path(candidate).resolve() for candidate in library_candidates
        if Path(candidate).name.startswith("libEGL_nvidia.so.") and Path(candidate).is_file()
    ), None)
    if library is None:
        raise RuntimeError(
            "No preinstalled NVIDIA EGL library was found in the standard driver paths. "
            "Select a compatible GPU runtime; this notebook will not use software rendering."
        )
    vendor_config = Path(session) / "nvidia-egl.json"
    vendor_config.write_text(json.dumps({
        "file_format_version": "1.0.0", "ICD": {"library_path": str(library)},
    }, indent=2) + "\n", encoding="utf-8")
    os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = str(vendor_config.resolve())
    driver_directory = str(library.parent)
    existing = [
        entry for entry in os.environ.get("LD_LIBRARY_PATH", "").split(":")
        if entry and entry != driver_directory
    ]
    os.environ["LD_LIBRARY_PATH"] = ":".join([driver_directory, *existing])
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYOPENGL_PLATFORM"] = "egl"
    return library

NVIDIA_EGL = configure_nvidia_egl(SESSION, (
    "/usr/lib64-nvidia/libEGL_nvidia.so.0",
    "/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0",
    "/usr/lib/x86_64-linux-gnu/nvidia/current/libEGL_nvidia.so.0",
    "/usr/lib/nvidia/libEGL_nvidia.so.0",
))
print("Session NVIDIA EGL library:", NVIDIA_EGL)

# A fresh isolated child must verify the actual GL vendor, not just library presence.
graphics_probe = r"""import json
import os
from pathlib import Path
import sys
import time
import mujoco
import numpy as np
from OpenGL import GL

model = mujoco.MjModel.from_xml_string(
    '<mujoco><visual><global offwidth="64" offheight="64"/></visual>'
    '<worldbody><light pos="0 0 3"/>'
    '<geom type="plane" size="1 1 .1" rgba=".3 .3 .3 1"/>'
    '<geom type="box" pos="0 0 .12" size=".1 .1 .12" rgba=".8 .2 .1 1"/>'
    '</worldbody></mujoco>'
)
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
renderer = mujoco.Renderer(model, height=64, width=64)
try:
    renderer.update_scene(data)
    started = time.perf_counter()
    pixels = renderer.render()
    render_ms = (time.perf_counter() - started) * 1000
    vendor = (GL.glGetString(GL.GL_VENDOR) or b"").decode("utf-8", "replace")
    device = (GL.glGetString(GL.GL_RENDERER) or b"").decode("utf-8", "replace")
    nonzero_pixels = int(np.count_nonzero(np.any(pixels != 0, axis=-1)))
    if "NVIDIA" not in vendor.upper() or not device:
        raise RuntimeError(
            f"NVIDIA EGL rendering is required; actual GL vendor={vendor!r}, renderer={device!r}"
        )
    if pixels.shape != (64, 64, 3) or nonzero_pixels == 0:
        raise RuntimeError("NVIDIA EGL did not produce nonempty 64x64 RGB pixels")
    graphics = {
        "gl_vendor": vendor, "gl_renderer": device, "shape": list(pixels.shape),
        "nonzero_pixels": nonzero_pixels, "render_ms": render_ms,
        "egl_vendor_config": os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"],
    }
    Path(sys.argv[1]).write_text(json.dumps(graphics, indent=2) + "\n", encoding="utf-8")
    print("[PASS] actual NVIDIA EGL rendering:", json.dumps(graphics), flush=True)
finally:
    renderer.close()
"""
run([PYTHON, "-c", graphics_probe, SESSION / "graphics.json"], cwd=REPO)
graphics = json.loads((SESSION / "graphics.json").read_text())
print(json.dumps(graphics, indent=2))

run([
    PYTHON, "tools/colab.py", "check", "--repo", REPO,
    "--out", SESSION / "runtime.json",
], cwd=REPO)
runtime = json.loads((SESSION / "runtime.json").read_text())
print(json.dumps(runtime, indent=2))
for package, expected in CORE_VERSIONS.items():
    actual = runtime["versions"][package]
    if actual != expected:
        raise RuntimeError(f"Core version restore failed: {package} expected {expected}, got {actual}")
run([PYTHON, "scripts/train.py", "--list"], cwd=REPO)
RUN_STATE["runtime"] = runtime

# Detect an older replay CLI before attempting to train anything.
play_help = subprocess.check_output(
    [str(PYTHON), "scripts/play.py", "--help"], cwd=REPO, text=True,
)
required_video_flags = ("--video", "--video-fps", "--video-width", "--video-height")
if any(flag not in play_help for flag in required_video_flags):
    raise RuntimeError("This source revision lacks the notebook's replay video interface")
required_replay_flags = ("--checkpoint-command-ranges", "--command", "--evaluation-out", "--replay-info-out")
if any(flag not in play_help for flag in required_replay_flags):
    raise RuntimeError("This source predates the improved evaluation workflow; use its matching older notebook for continuation.")


## 4. Restore checks and optional checkpoint persistence

Continuation fully checks the archive, restores it into a new directory and validates the checkpoint against this source/runtime before training.
Drive snapshots, when enabled, publish complete checkpoints during training. They require your Drive authorization.


In [ ]:
try:
    # @title Verify the selected continuation checkpoint
    RESTORED_CHECKPOINT = RUN_STATE.get("restored_checkpoint")
    if WORKFLOW_MODE == "Continue training":
        archive = RUN_STATE.get("backup_archive")
        if archive and not RESTORED_CHECKPOINT:
            restored = SESSION / ("restored-" + uuid.uuid4().hex)
            restore_info = SESSION / "restore.json"
            run([PYTHON, "tools/colab.py", "restore", "--archive", archive,
                 "--destination", restored, "--out", restore_info], cwd=REPO)
            RESTORED_CHECKPOINT = json.loads(restore_info.read_text())["checkpoint"]
            RUN_STATE["restored_checkpoint"] = RESTORED_CHECKPOINT
        if not RESTORED_CHECKPOINT or not Path(RESTORED_CHECKPOINT).is_file():
            raise RuntimeError("Continuation requires the explicitly restored checkpoint")
        from textwrap import dedent

        verify_resume = dedent("""
    import importlib.util,json,sys
    from pathlib import Path
    spec=importlib.util.spec_from_file_location('colab',Path(sys.argv[1])/'tools/colab.py')
    colab=importlib.util.module_from_spec(spec);spec.loader.exec_module(colab)
    cfg=colab.make_config(Path(sys.argv[1]),task=sys.argv[2],model=sys.argv[3],num_envs=int(sys.argv[4]))
    colab.validate_resume(Path(sys.argv[5]),cfg)
    print(json.dumps({'mode':'Continue training','checkpoint':sys.argv[5],'config':cfg},indent=2))
    """)
        run([PYTHON, "-c", verify_resume, REPO, TASK, MODEL, str(NUM_ENVS), RESTORED_CHECKPOINT], cwd=REPO)
    else:
        print("New training: no continuation checkpoint selected.")
except BaseException:
    previous = RUN_STATE.pop("previous_backup_selection", None)
    if previous:
        RUN_STATE.update(previous)
        RESTORED_CHECKPOINT = RUN_STATE.get("restored_checkpoint")
        print("Restore failed; the previous backup selection is retained. Re-run environment matching before continuing it.")
    raise
else:
    if WORKFLOW_MODE == "Continue training":
        RUN_STATE.pop("previous_backup_selection", None)


In [ ]:
PERSIST_ROOT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not DRIVE_FOLDER or Path(DRIVE_FOLDER).is_absolute() or ".." in Path(DRIVE_FOLDER).parts:
        raise ValueError("DRIVE_FOLDER must be a relative folder under MyDrive")
    PERSIST_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
    print("Complete checkpoint snapshots will be saved under:", PERSIST_ROOT)
else:
    print("Drive is off. Results remain in /content until you download them.")


## 5. Optional demonstration and short test

The dance uses a supplied policy and is independent of your training result.
The smoke run uses your chosen task and at most your selected environment count (capped at 256), for five updates. It never changes the continuation checkpoint.
Set RUN_DEMO/RUN_SMOKE in Advanced source, or skip these cells.


In [ ]:
if RUN_DEMO:
    from IPython.display import Image, Video, display

    DEMO = REPO / "tasks/jumper/dance/out/example/model_2300.pt"
    if not DEMO.is_file():
        raise FileNotFoundError(f"Bundled pretrained demo checkpoint is missing: {DEMO}")
    DEMO_VIDEO = SESSION / "pretrained-dance.mp4"
    run([
        PYTHON, "scripts/play.py", "--task", "jumper.dance", "--model", MODEL,
        "--checkpoint", DEMO, "--backend", "warp", "--device", "cuda:0",
        "--num_envs", "1", "--headless", "--steps", "500", "--physics-hz", "200",
        "--video", DEMO_VIDEO, "--video-fps", "30",
        "--video-width", "640", "--video-height", "480", "--strip-visual", "off",
    ], cwd=REPO)
    if not DEMO_VIDEO.is_file() or DEMO_VIDEO.stat().st_size == 0:
        raise RuntimeError("Replay did not produce the demo MP4")
    display(Video(str(DEMO_VIDEO), embed=True))
else:
    print("Pretrained demo skipped.")


In [ ]:
if RUN_SMOKE:
    SMOKE_MANIFEST = SESSION / "smoke.json"
    run([
        PYTHON, "tools/colab.py", "train", "--repo", REPO,
        "--task", TASK, "--model", MODEL, "--num-envs", str(min(NUM_ENVS, 256)),
        "--iterations", "5", "--manifest", SMOKE_MANIFEST,
    ], cwd=REPO)
    smoke = json.loads(SMOKE_MANIFEST.read_text())
    if smoke["status"] != "finished":
        raise RuntimeError(f"Smoke test failed: {smoke}")
    print("GPU smoke test completed:", smoke["run_directory"])
else:
    print("GPU smoke test skipped.")


## 6. Train or continue

The cell prints the mode, checkpoint and settings before starting. Continuation adds ITERATIONS updates to the saved learning state.
The checkpoint restores the networks, optimizer and curricula; simulation episodes start again. Saved checkpoints remain available after an interrupted run.


In [ ]:
EXPORT_SUCCESS = False
EXPORTED_CHECKPOINT = ""
SIMULATION_CHECKPOINT = ""
MEASUREMENT_CHECKPOINT = ""
TRAIN_MANIFEST = SESSION / "training.json"
train_command = [
    PYTHON, "tools/colab.py", "train", "--repo", REPO,
    "--task", TASK, "--model", MODEL, "--num-envs", str(NUM_ENVS),
    "--iterations", str(ITERATIONS), "--manifest", TRAIN_MANIFEST,
]
if PERSIST_ROOT is not None:
    train_command += ["--drive-root", PERSIST_ROOT]
if WORKFLOW_MODE == "Continue training":
    selected_checkpoint = RUN_STATE.get("restored_checkpoint")
    if not selected_checkpoint:
        raise RuntimeError("Restore and verify a checkpoint before continuing")
    recorded = RUN_STATE["backup_config"]
    if (TASK, MODEL, NUM_ENVS) != (recorded["task"], recorded["model"], recorded["num_envs"]):
        raise ValueError("Continuation task/model/environment count must match the restored run")
    train_command += ["--resume", Path(selected_checkpoint)]
elif RUN_STATE.get("backup_config"):
    raise RuntimeError("Recovered run is preserved; choose Continue training or explicitly START_NEW_RUN")
print("Training mode:", WORKFLOW_MODE, "checkpoint:", RUN_STATE.get("restored_checkpoint"),
      "task:", TASK, "environments:", NUM_ENVS, "updates:", ITERATIONS)
RUN_STATE["training_manifest"] = str(TRAIN_MANIFEST)
run(train_command, cwd=REPO)
training = json.loads(TRAIN_MANIFEST.read_text())
if training["status"] != "finished":
    raise RuntimeError(f"Training did not finish: {training}")
RUN_DIR = Path(training["run_directory"])
CHECKPOINT_PATH = Path(training["checkpoint"])
print("This run:", RUN_DIR)
print("Selected checkpoint:", CHECKPOINT_PATH)

# A separate attachment directory per run prevents a failed rerun from packaging old curves.
import shutil

ARTIFACTS = SESSION / "artifacts" / training["operation"]
ARTIFACTS.mkdir(parents=True, exist_ok=True)
for name in ("runtime.json", "graphics.json"):
    shutil.copy2(SESSION / name, ARTIFACTS / name)
print("Run attachments:", ARTIFACTS)


## 7. Check training progress

Inspect reward/loss curves together with the command curriculum and speed error versus its promotion threshold.
The summary lists the latest finite value and its iteration. Missing task-specific metrics remain unavailable; finishing the update budget is not gait acceptance.


In [ ]:
from IPython.display import Image, display

METRICS_JSON = ARTIFACTS / "metrics.json"
CURVES_PNG = ARTIFACTS / "training-curves.png"
run([
    PYTHON, "tools/colab.py", "metrics", "--run", RUN_DIR,
    "--out", METRICS_JSON, "--plot", CURVES_PNG,
], cwd=REPO)
metrics = json.loads(METRICS_JSON.read_text())
if Path(metrics["run"]).resolve() != RUN_DIR.resolve():
    raise RuntimeError("Training curves belong to a different run")
print("Full scalar history JSON:", METRICS_JSON)
print("Full scalar history CSV:", METRICS_JSON.with_suffix(".csv"))
display(Image(filename=str(CURVES_PNG)))

print("Latest recorded training progress:")
for tag, point in metrics.get("progress", {}).items():
    print(f"{tag}: {point['value']:.6g} (iteration {point['step']})")
if not metrics.get("progress"):
    print("This task did not record command-curriculum metrics; use its task-specific evaluation.")


## 8. Simulate and evaluate

Replay uses the selected checkpoint. Velocity tasks default to the command range reached by its saved curriculum; the replay report records that range.
Choose a sampled command or a fixed body-frame command. Fixed evaluations measure forward, sideways, turn and stand separately; changed terrain is a separate test condition.
To change a replay, rerun only this settings form and replay/evaluation cells. Keep the policy task/model unchanged.


In [ ]:
# @title Simulation and evaluation settings
SIM_SECONDS = 10.0  # @param {type:"number"}
SIM_COMMAND = "Sampled commands"  # @param ["Sampled commands", "Forward", "Sideways", "Turn", "Stand", "Custom"]
SIM_VX = 0.1  # @param {type:"number"}
SIM_VY = 0.1  # @param {type:"number"}
SIM_YAW = 0.2  # @param {type:"number"}
RUN_EVALUATION = True  # @param {type:"boolean"}
EVAL_SECONDS = 4.0  # @param {type:"number"}
RECORD_JOINTS = True  # @param {type:"boolean"}
SIM_FPS = 30.0  # @param {type:"number"}
SIM_WIDTH = 640  # @param {type:"integer"}
SIM_HEIGHT = 480  # @param {type:"integer"}
SIM_SCENE = "task default"  # @param ["task default", "default", "studio", "daylight", "beach", "football", "soft", "ice", "rubber", "plain", "rough"]
SIM_CAMERA_DISTANCE = 0.0  # @param {type:"number"}

if any(not math.isfinite(v) or v <= 0 for v in (SIM_SECONDS, SIM_FPS, EVAL_SECONDS)):
    raise ValueError("Simulation/evaluation duration and FPS must be finite and positive")
if any(isinstance(v, bool) or not isinstance(v, int) or v < 2 or v % 2 for v in (SIM_WIDTH, SIM_HEIGHT)):
    raise ValueError("SIM_WIDTH and SIM_HEIGHT must be positive even integers")
if not math.isfinite(SIM_CAMERA_DISTANCE) or SIM_CAMERA_DISTANCE < 0:
    raise ValueError("Camera distance must be zero (automatic) or positive")
if any(not math.isfinite(v) for v in (SIM_VX, SIM_VY, SIM_YAW)):
    raise ValueError("Fixed command components must be finite")
SIM_COMMANDS = {"Forward": [SIM_VX, 0.0, 0.0], "Sideways": [0.0, SIM_VY, 0.0],
                "Turn": [0.0, 0.0, SIM_YAW], "Stand": [0.0, 0.0, 0.0],
                "Custom": [SIM_VX, SIM_VY, SIM_YAW]}
if SIM_COMMAND not in ("Sampled commands", *SIM_COMMANDS):
    raise ValueError("Choose an available command mode")
if SIM_SCENE not in ("task default", "default", "studio", "daylight", "beach", "football",
                     "soft", "ice", "rubber", "plain", "rough"):
    raise ValueError("Choose an available replay scene")


In [ ]:
# Replay settings can change without retraining; the policy task/model cannot.
recorded_config = training["config"]
if TASK != recorded_config["task"] or MODEL != recorded_config["model"]:
    raise ValueError(
        "The selected checkpoint belongs to a different task/model. "
        "Change only SIM_* or RECORD_JOINTS for replay; train a new run to change TASK/MODEL."
    )

if Path(CHECKPOINT_PATH).resolve() != Path(training["checkpoint"]).resolve():
    raise ValueError("Replay checkpoint must match the selected training manifest")

# Preserve previous telemetry without putting it beside the new recording in the ZIP.
previous_outputs = [(CHECKPOINT_PATH.parent / "measure", "checkpoint-measure")]
previous_outputs += [(ARTIFACTS / name, name) for name in (
    "measure.csv", "measure.png", "simulation-settings.json",
)]
REPLAY_HISTORY = SESSION / "replay-history" / uuid.uuid4().hex
for previous, name in previous_outputs:
    if previous.is_symlink():
        raise RuntimeError(f"Replay refuses a symlink at its generated output: {previous}")
    if previous.exists():
        REPLAY_HISTORY.mkdir(parents=True, exist_ok=True)
        shutil.move(str(previous), str(REPLAY_HISTORY / name))

rates_file = SESSION / "rates.json"
run([
    PYTHON, "tools/colab.py", "rates", "--repo", REPO,
    "--task", TASK, "--model", MODEL, "--out", rates_file,
], cwd=REPO)
rates = json.loads(rates_file.read_text())
SIM_STEPS = math.ceil(SIM_SECONDS * rates["control_hz"])
print("Replay rates:", rates, "control steps:", SIM_STEPS)
SIMULATION_CHECKPOINT = ""
MEASUREMENT_CHECKPOINT = ""
TRAIN_VIDEO = SESSION / f"trained-policy-{training['operation']}-{uuid.uuid4().hex[:8]}.mp4"
replay_command = [
    PYTHON, "scripts/play.py", "--task", TASK, "--model", MODEL,
    "--checkpoint", CHECKPOINT_PATH, "--backend", "warp", "--device", "cuda:0",
    "--num_envs", "1", "--headless", "--steps", str(SIM_STEPS),
    "--physics-hz", str(rates["physics_hz"]), "--video", TRAIN_VIDEO,
    "--video-fps", str(SIM_FPS), "--video-width", str(SIM_WIDTH),
    "--video-height", str(SIM_HEIGHT), "--strip-visual", "off",
]
REPLAY_INFO = ARTIFACTS / "replay-info.json"
replay_command += ["--replay-info-out", REPLAY_INFO]
if rates["supports_velocity_command"]:
    replay_command += ["--checkpoint-command-ranges"]
    if SIM_COMMAND != "Sampled commands":
        replay_command += ["--command", *map(str, SIM_COMMANDS[SIM_COMMAND])]
elif SIM_COMMAND != "Sampled commands":
    raise ValueError("This task has no supported velocity command; choose Sampled commands")
if SIM_SCENE != "task default":
    replay_command += ["--scene", SIM_SCENE]
if SIM_CAMERA_DISTANCE > 0:
    replay_command += ["--video-distance", str(SIM_CAMERA_DISTANCE)]
if RECORD_JOINTS:
    replay_command += ["--measure"]
run(replay_command, cwd=REPO)
if not TRAIN_VIDEO.is_file() or TRAIN_VIDEO.stat().st_size == 0:
    raise RuntimeError("Replay did not produce the trained MP4")
SIMULATION_CHECKPOINT = str(CHECKPOINT_PATH)
replay_info = json.loads(REPLAY_INFO.read_text())
simulation_settings = {
    "checkpoint": str(CHECKPOINT_PATH), "run": str(RUN_DIR), "scene": SIM_SCENE,
    "physics_hz": rates["physics_hz"], "control_hz": rates["control_hz"],
    "requested_seconds": SIM_SECONDS, "simulated_seconds": SIM_STEPS / rates["control_hz"],
    "control_steps": SIM_STEPS, "fps": SIM_FPS, "width": SIM_WIDTH, "height": SIM_HEIGHT,
    "camera_distance": SIM_CAMERA_DISTANCE, "observation_noise": True,
    "record_joints": RECORD_JOINTS, "command_mode": SIM_COMMAND,
    "replay_info": replay_info,
}
(ARTIFACTS / "simulation-settings.json").write_text(
    json.dumps(simulation_settings, indent=2) + "\n", encoding="utf-8",
)
replay_info = json.loads(REPLAY_INFO.read_text())
print("Replay command range:", replay_info.get("command_ranges"))
from IPython.display import Image, Video, display

display(Video(str(TRAIN_VIDEO), embed=True))
if RECORD_JOINTS:
    measurement_directory = CHECKPOINT_PATH.parent / "measure"
    for name in ("measure.csv", "measure.png"):
        source = measurement_directory / name
        if not source.is_file() or source.stat().st_size == 0:
            raise RuntimeError(f"Requested joint recording is missing: {source}")
        shutil.copy2(source, ARTIFACTS / name)
    MEASUREMENT_CHECKPOINT = str(CHECKPOINT_PATH)
    print("Joint telemetry CSV:", ARTIFACTS / "measure.csv")
    display(Image(filename=str(ARTIFACTS / "measure.png")))


In [ ]:
# @title Fixed-command evaluation
from IPython.display import display

if RUN_EVALUATION and rates["supports_velocity_command"]:
    evaluation_training = json.loads(TRAIN_MANIFEST.read_text())
    if (TASK, MODEL) != (evaluation_training["config"]["task"], evaluation_training["config"]["model"]):
        raise ValueError("Evaluation task/model must match the selected training run")
    if Path(CHECKPOINT_PATH).resolve() != Path(evaluation_training["checkpoint"]).resolve():
        raise ValueError("Evaluation checkpoint must match the selected training manifest")
    EVALUATION_DIR = ARTIFACTS / ("evaluation-" + uuid.uuid4().hex[:8])
    EVALUATION_DIR.mkdir()
    EVALUATION_RESULTS = []
    fixed_cases = {"forward": [SIM_VX, 0.0, 0.0], "sideways": [0.0, SIM_VY, 0.0],
                   "turn": [0.0, 0.0, SIM_YAW], "stand": [0.0, 0.0, 0.0]}
    for label, command in fixed_cases.items():
        output = EVALUATION_DIR / (label + ".json")
        evaluation_command = [PYTHON, "scripts/play.py", "--task", TASK, "--model", MODEL,
            "--checkpoint", CHECKPOINT_PATH, "--backend", "warp", "--device", "cuda:0",
            "--num_envs", "1", "--headless", "--steps", str(math.ceil(EVAL_SECONDS * rates["control_hz"])),
            "--physics-hz", str(rates["physics_hz"]), "--checkpoint-command-ranges",
            "--command", *map(str, command), "--evaluation-out", output]
        if SIM_SCENE != "task default":
            evaluation_command += ["--scene", SIM_SCENE]
        run(evaluation_command, cwd=REPO)
        result = json.loads(output.read_text())
        result["case"] = label
        result["scene"] = SIM_SCENE
        output.write_text(json.dumps(result, indent=2) + "\n")
        EVALUATION_RESULTS.append(result)
        print(label + ":", json.dumps(result))
    (EVALUATION_DIR / "summary.json").write_text(json.dumps(EVALUATION_RESULTS, indent=2) + "\n")
    print("Fixed-command measurements:", EVALUATION_DIR)
else:
    print("Fixed velocity evaluation skipped:" , "disabled" if not RUN_EVALUATION else "not supported by this task")


In [ ]:
from google.colab import files

if DOWNLOAD_FILES:
    if globals().get("RUN_DEMO", False) and globals().get("DEMO_VIDEO") and DEMO_VIDEO.is_file():
        files.download(str(DEMO_VIDEO))
    files.download(str(TRAIN_VIDEO))
    if MEASUREMENT_CHECKPOINT == str(CHECKPOINT_PATH):
        files.download(str(ARTIFACTS / "measure.csv"))
else:
    print("Automatic downloads are off. Videos and telemetry remain in the session and backup.")


## 9. Export and keep a backup

Export includes the ONNX Actor and layout contract. Backup preserves complete checkpoints, configuration, logs and available results from this run.
The backup cell can run after a failed training, plot, replay or export cell if at least one complete checkpoint was saved. Missing optional graphics reports do not prevent it.


In [ ]:
EXPORT_SUCCESS = False
EXPORTED_CHECKPOINT = ""
recorded_config = training["config"]
if TASK != recorded_config["task"] or MODEL != recorded_config["model"]:
    raise ValueError(
        "The selected checkpoint belongs to a different task/model. "
        "Keep TASK/MODEL unchanged for export; train a new run to change them."
    )

EXPORT_DIR = SESSION / f"onnx-export-{uuid.uuid4().hex[:8]}"
run([
    PYTHON, "scripts/export.py", "--task", TASK, "--model", MODEL,
    "--checkpoint", CHECKPOINT_PATH, "--backend", "warp", "--device", "cuda:0",
    "--num_envs", "1", "--no-video", "--out", EXPORT_DIR,
], cwd=REPO)
if not (EXPORT_DIR / "actor.onnx").is_file() or not (EXPORT_DIR / "layout.json").is_file():
    raise RuntimeError("ONNX export did not produce its policy and contract")
print("ONNX policy and contract:", EXPORT_DIR)
EXPORTED_CHECKPOINT = str(CHECKPOINT_PATH)
EXPORT_SUCCESS = True
(ARTIFACTS / "export-settings.json").write_text(json.dumps({
    "checkpoint": EXPORTED_CHECKPOINT, "directory": str(EXPORT_DIR), "run": str(RUN_DIR),
}, indent=2) + "\n", encoding="utf-8")


In [ ]:
import shutil

from google.colab import files

# The manifest selects this run even when curves, replay or export failed later.
TRAIN_MANIFEST = Path(globals().get("RUN_STATE", {}).get("training_manifest", globals().get("TRAIN_MANIFEST", SESSION / "training.json")))
training = json.loads(TRAIN_MANIFEST.read_text())
if not training.get("run_directory") or not training.get("checkpoints"):
    raise RuntimeError("No complete training checkpoint is available to back up")
RUN_DIR = Path(training["run_directory"])
CHECKPOINT_PATH = Path(training["checkpoint"])
BACKUP_CHECKPOINT = str(CHECKPOINT_PATH)
ARTIFACTS = SESSION / "artifacts" / training["operation"]
ARTIFACTS.mkdir(parents=True, exist_ok=True)
for name in ("runtime.json", "graphics.json"):
    if (SESSION / name).is_file() and not (ARTIFACTS / name).exists():
        shutil.copy2(SESSION / name, ARTIFACTS / name)
BACKUP_ZIP = SESSION / f"jumper-{TASK.replace('.', '-')}-{uuid.uuid4().hex[:8]}.zip"
backup_command = [
    PYTHON, "tools/colab.py", "backup", "--run", RUN_DIR, "--out", BACKUP_ZIP,
    "--extras", ARTIFACTS,
]
if (globals().get("EXPORT_SUCCESS", False)
        and globals().get("EXPORTED_CHECKPOINT") == BACKUP_CHECKPOINT):
    backup_command += ["--export", EXPORT_DIR]
if (globals().get("SIMULATION_CHECKPOINT") == BACKUP_CHECKPOINT
        and TRAIN_VIDEO.is_file() and TRAIN_VIDEO.stat().st_size):
    backup_command += ["--video", TRAIN_VIDEO]
run(backup_command, cwd=REPO)
print("Portable backup:", BACKUP_ZIP, "bytes:", BACKUP_ZIP.stat().st_size)
if DOWNLOAD_FILES:
    files.download(str(BACKUP_ZIP))
else:
    print("Download this ZIP before Colab discards the runtime:", BACKUP_ZIP)


## Interruption and troubleshooting

- **Continuation:** use Continue training; read the backup first. Re-running settings preserves its checkpoint. To deliberately start anew, select New training and START_NEW_RUN.
- **Training interrupted:** run the backup cell for already complete checkpoints. A lost runtime cannot copy files afterwards; use a downloaded ZIP or a completed Drive snapshot.
- **Source mismatch:** continuation uses the recorded commit and core versions. If an old commit lacks this notebook's replay features, use its matching older notebook; do not change provenance to force a load.
- **GPU memory:** reduce NUM_ENVS for a new run; continuation requires its recorded batch. Optional smoke uses at most the selected count.
- **Unsupported fixed command:** use sampled replay for non-velocity tasks. Velocity evaluation refuses commands outside the saved training range.
- **Render/export failed:** complete training checkpoints can still be backed up. Missing results are not successful evaluations.

Historical T4 validation and current workflow verification are recorded separately in the README and guide. Drive mounting and physical-robot behavior require their own checks.
